In [1]:
import os, sys
os.chdir(os.path.dirname(sys.prefix))

In [6]:
import glob
import pymupdf
import random
from tqdm import tqdm
from pathlib import Path

In [3]:
pdf_files = glob.glob('data/pdf/*.pdf')
len(pdf_files)

35

In [4]:
total_pages = 0
total_memory = 0  # в байтах

for pdf_file in pdf_files:
    with pymupdf.open(pdf_file) as doc:
        total_pages += doc.page_count
    total_memory += os.path.getsize(pdf_file)

print(f"Общее число страниц во всех pdf файлах: {total_pages}")
print(f"Общий объём памяти, занимаемый pdf файлами: {total_memory / (1024 * 1024):.2f} МБ")

Общее число страниц во всех pdf файлах: 34123
Общий объём памяти, занимаемый pdf файлами: 784.96 МБ


In [10]:
name_temp = "generated_{:05}.png"
save_dir = Path("./data/generated/v3")
os.makedirs(save_dir, exist_ok=True)
save_dir / name_temp.format(0)

PosixPath('data/generated/v3/generated_00000.png')

In [11]:
random.seed(52)
idx = 0

# Сначала посчитаем сколько всего будет изображений для правильного tqdm
total_images = 0
for pdf_file in pdf_files:
    with pymupdf.open(pdf_file) as doc:
        n_pages = doc.page_count
        total_images += max(n_pages - 10, 10) - 10

with tqdm(total=total_images, desc="Генерация изображений") as pbar:
    for pdf_file in pdf_files:
        with pymupdf.open(pdf_file) as doc:
            n_pages = doc.page_count
            # Пропускаем первые 10 и последние 10 страниц
            for pg_num in range(10, max(n_pages - 10, 10)):
                page = doc[pg_num]
                # Случайно выбрать целое от 100 до 200, чаще ближе к 100 (распределение экспоненциальное сдвинутое)
                dpi = int(100 + random.expovariate(1/20))
                dpi = min(dpi, 200)
                pix = page.get_pixmap(dpi=dpi)
                name = name_temp.format(idx)
                save_path = save_dir / name
                pix.save(str(save_path))
                idx += 1
                pbar.update(1)

Генерация изображений:   8%|▊         | 2684/33428 [15:11<2:54:02,  2.94it/s] 


KeyboardInterrupt: 